# SSL Vision Foundation — GPU runner (Colab / Kaggle)
1. Runtime → GPU (T4/L4/A100).
2. Colab: add secret `HF_TOKEN` (key icon, left sidebar). Kaggle: Add-ons → Secrets → `HF_TOKEN`.
3. Every run checkpoints to `ahmedashrafk12/ssl-vision-foundation` on the Hub, so a disconnect just means re-running the same cell — it resumes.

In [ ]:
import os
try:
    from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except ImportError:
    from kaggle_secrets import UserSecretsClient; os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_REPO'] = 'ahmedashrafk12/ssl-vision-foundation'
REPO_URL = 'https://github.com/ahmedashrafk13/ssl-vision-foundation.git'
!git clone -q $REPO_URL ssl-vision-foundation || true
%cd ssl-vision-foundation
!pip -q install -r requirements.txt
!python -m pytest -q tests

In [ ]:
# data (MVTec ~5GB unpacked, ImageNet-100 subset 30k imgs at 300/class). CIFAR downloads on demand.
!python scripts/download_data.py mvtec
!python scripts/download_data.py imagenet100 --per-class 300
# NEU-DET: upload kaggle.json first, or on Kaggle attach the dataset and symlink it to data/neu-det
!python scripts/download_data.py neu

In [ ]:
# quick sanity run (~2 min) before burning hours
!python scripts/pretrain.py --method mae --backbone vit_tiny --img 224 --epochs 1 --max-steps 20 --out sanity_mae
!python scripts/finetune.py --task seg --mode probe --backbone vit_tiny --ckpt sanity_mae/backbone.pt --epochs 1 --max-steps 20

In [ ]:
# pre-training — run one per session if you're on free tier; each resumes from the Hub
!python scripts/pretrain.py --method simclr --backbone resnet50 --img 224 --epochs 100 --bs 256
!python scripts/pretrain.py --method mae --backbone vit_small --img 224 --epochs 200 --bs 256

In [ ]:
# full grid: probes, fine-tunes at 1/10/25/50/100% labels, DETR head check, mask ablation, export bench
!bash scripts/run_all.sh

In [ ]:
!python scripts/plot_results.py
from huggingface_hub import HfApi
HfApi().upload_folder(folder_path='results', path_in_repo='results', repo_id=os.environ['HF_REPO'])